# Spiral-PRIME Quick Start Guide

This notebook demonstrates basic usage of the Spiral-PRIME reconstruction pipeline.

## Overview

Spiral-PRIME provides:
- Spiral trajectory design
- Coil sensitivity estimation
- Iterative SENSE reconstruction
- Advanced regularization (TV, L1, L2)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from spiral_prime_gpi.core import spiral_physics, mri_math, optimization

# Set random seed for reproducibility
np.random.seed(42)

## 1. Design a Spiral Trajectory

First, we'll design a 2D spiral trajectory with specified field-of-view and resolution.

In [ ]:
# Trajectory parameters
fov = 24.0  # cm
resolution = 0.2  # cm (2 mm)
n_interleaves = 8

# Design trajectory
trajectory, gradient, time = spiral_physics.design_spiral_trajectory(
    fov=fov,
    resolution=resolution,
    n_interleaves=n_interleaves,
    gmax=40.0,  # mT/m
    smax=150.0  # T/m/s
)

print(f"Trajectory: {trajectory.shape[0]} points")
print(f"Readout duration: {time[-1]*1000:.2f} ms")

In [ ]:
# Visualize trajectory
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot k-space trajectory
axes[0].plot(trajectory[:, 0], trajectory[:, 1], 'b-', linewidth=0.5)
axes[0].plot(trajectory[0, 0], trajectory[0, 1], 'go', markersize=8, label='Start')
axes[0].plot(trajectory[-1, 0], trajectory[-1, 1], 'ro', markersize=8, label='End')
axes[0].set_xlabel('kx (normalized)')
axes[0].set_ylabel('ky (normalized)')
axes[0].set_title('Spiral k-space Trajectory')
axes[0].axis('equal')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot gradient waveforms
axes[1].plot(time * 1000, gradient[:, 0], label='Gx')
axes[1].plot(time * 1000, gradient[:, 1], label='Gy')
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Gradient (mT/m)')
axes[1].set_title('Gradient Waveforms')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Create a Phantom and Simulate Acquisition

We'll create a simple Shepp-Logan phantom and simulate spiral acquisition.

In [ ]:
def create_shepp_logan_phantom(size=128):
    """Create a simple Shepp-Logan-like phantom."""
    phantom = np.zeros((size, size))
    
    # Create meshgrid
    y, x = np.meshgrid(np.linspace(-1, 1, size), np.linspace(-1, 1, size))
    
    # Main ellipse
    phantom += ((x/0.69)**2 + (y/0.92)**2 < 1).astype(float) * 1.0
    
    # Two smaller ellipses (ventricles)
    phantom -= ((((x+0.22)/0.11)**2 + (y/0.31)**2) < 1).astype(float) * 0.8
    phantom -= ((((x-0.22)/0.16)**2 + (y/0.41)**2) < 1).astype(float) * 0.8
    
    # Small circular features
    phantom += ((x+0.35)**2 + (y+0.1)**2 < 0.046**2).astype(float) * 0.3
    phantom += ((x+0.1)**2 + (y+0.1)**2 < 0.046**2).astype(float) * 0.3
    
    return phantom

# Create phantom
image_size = (128, 128)
phantom = create_shepp_logan_phantom(image_size[0])

plt.figure(figsize=(6, 6))
plt.imshow(phantom, cmap='gray')
plt.title('Shepp-Logan Phantom')
plt.colorbar()
plt.axis('off')
plt.show()

In [ ]:
# Create synthetic coil sensitivity maps
n_coils = 8

def create_coil_maps(image_shape, n_coils):
    """Create synthetic coil sensitivity maps."""
    ny, nx = image_shape
    y, x = np.meshgrid(np.linspace(-1, 1, ny), np.linspace(-1, 1, nx))
    
    coil_maps = np.zeros((n_coils, ny, nx), dtype=complex)
    
    for c in range(n_coils):
        # Angle for this coil
        angle = 2 * np.pi * c / n_coils
        
        # Coil position
        cx = 1.5 * np.cos(angle)
        cy = 1.5 * np.sin(angle)
        
        # Sensitivity pattern (Gaussian-like)
        dist = np.sqrt((x - cx)**2 + (y - cy)**2)
        sensitivity = np.exp(-dist**2 / 2.0)
        
        # Add phase variation
        phase = angle + 0.1 * (x + y)
        
        coil_maps[c] = sensitivity * np.exp(1j * phase)
    
    # Normalize
    sos = np.sqrt(np.sum(np.abs(coil_maps)**2, axis=0))
    coil_maps = coil_maps / sos[np.newaxis, ...]
    
    return coil_maps

coil_maps = create_coil_maps(image_size, n_coils)

# Visualize coil maps
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for c in range(n_coils):
    ax = axes[c // 4, c % 4]
    ax.imshow(np.abs(coil_maps[c]), cmap='gray')
    ax.set_title(f'Coil {c+1}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Simulate k-space acquisition
kspace_data = spiral_physics.simulate_spiral_acquisition(
    phantom.astype(complex),
    trajectory,
    coil_maps,
    noise_std=0.01  # Add some noise
)

print(f"K-space data shape: {kspace_data.shape}")
print(f"Data type: {kspace_data.dtype}")

## 3. Perform Reconstruction

Now we'll reconstruct the image using iterative SENSE.

In [ ]:
# Compute density compensation
density_comp = mri_math.compute_density_compensation(trajectory, method='voronoi')

# Reconstruct without regularization
print("Reconstructing without regularization...")
recon_no_reg = optimization.iterative_sense_recon(
    kspace_data,
    trajectory,
    coil_maps,
    image_shape=image_size,
    n_iterations=10,
    regularization_type='none',
    density_comp=density_comp
)

# Reconstruct with TV regularization
print("Reconstructing with TV regularization...")
recon_tv = optimization.iterative_sense_recon(
    kspace_data,
    trajectory,
    coil_maps,
    image_shape=image_size,
    n_iterations=10,
    regularization_type='tv',
    lambda_reg=0.001,
    density_comp=density_comp
)

print("Reconstruction complete!")

In [ ]:
# Display results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original phantom
axes[0].imshow(phantom, cmap='gray')
axes[0].set_title('Original Phantom')
axes[0].axis('off')

# Reconstruction without regularization
im1 = axes[1].imshow(np.abs(recon_no_reg), cmap='gray')
axes[1].set_title('Reconstruction (No Regularization)')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

# Reconstruction with TV
im2 = axes[2].imshow(np.abs(recon_tv), cmap='gray')
axes[2].set_title('Reconstruction (TV Regularization)')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()

## 4. Compare Reconstruction Quality

Let's compute some metrics to quantify reconstruction quality.

In [ ]:
def compute_nrmse(reference, reconstruction):
    """Compute normalized root mean squared error."""
    reference = np.abs(reference)
    reconstruction = np.abs(reconstruction)
    
    mse = np.mean((reference - reconstruction)**2)
    nrmse = np.sqrt(mse) / np.mean(reference)
    return nrmse

# Compute errors
nrmse_no_reg = compute_nrmse(phantom, recon_no_reg)
nrmse_tv = compute_nrmse(phantom, recon_tv)

print(f"NRMSE without regularization: {nrmse_no_reg:.4f}")
print(f"NRMSE with TV regularization: {nrmse_tv:.4f}")
print(f"\nImprovement: {(nrmse_no_reg - nrmse_tv) / nrmse_no_reg * 100:.2f}%")

## Summary

This notebook demonstrated:

1. **Spiral trajectory design** with configurable FOV, resolution, and gradient constraints
2. **Coil sensitivity map** creation and visualization
3. **K-space data simulation** from a phantom image
4. **Iterative SENSE reconstruction** with and without regularization
5. **Quality assessment** using NRMSE metrics

## Next Steps

- Try different regularization parameters
- Experiment with 3D stack-of-spirals trajectories
- Use real MRI data
- Explore the GPI graphical interface

For more information, see the [documentation](../docs/) and [API reference](../docs/api_reference.md).